In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
df_temperature = pd.read_csv("Source_Data/temperaturdaten.csv", encoding="utf-8", sep=";")
df_fitness =  pd.read_csv("Dataframes/workouts_per_second.csv", encoding="utf-8")
df_session = pd.read_csv("Dataframes/session_records.csv", encoding="utf-8")

In [3]:
df_session.columns

Index(['source_file', 'message_type', 'avg_cadence', 'avg_heart_rate',
       'avg_power', 'avg_speed', 'enhanced_avg_speed', 'enhanced_max_speed',
       'event', 'event_group', 'event_type', 'first_lap_index', 'max_cadence',
       'max_heart_rate', 'max_power', 'max_speed', 'message_index', 'nec_lat',
       'nec_long', 'num_laps', 'sport', 'start_position_lat',
       'start_position_long', 'start_time', 'sub_sport', 'swc_lat', 'swc_long',
       'timestamp', 'total_ascent', 'total_calories', 'total_cycles',
       'total_descent', 'total_distance', 'total_elapsed_time',
       'total_fat_calories', 'total_timer_time', 'total_training_effect',
       'trigger', 'unknown_193'],
      dtype='str')

In [4]:
df_session.dropna(axis=1, how='all', inplace = True)
df_session.describe(include = 'all')

,source_file,message_type,avg_cadence,avg_heart_rate,avg_power,avg_speed,enhanced_avg_speed,enhanced_max_speed,event,event_group,...,total_calories,total_cycles,total_descent,total_distance,total_elapsed_time,total_fat_calories,total_timer_time,total_training_effect,trigger,unknown_193
count,244,244,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244,244.0,...,244.000000,244.0,244.0,244.000000,244.000000,244.0,244.000000,244.0,244,4.000000
unique,244,1,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
top,2026-01-10-13-32-12.fit,session,NaN,NaN,NaN,NaN,NaN,NaN,session,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,activity_end,NaN
freq,1,244,NaN,NaN,NaN,NaN,NaN,NaN,244,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,244,NaN
mean,NaN,NaN,53.909836,85.368852,85.077869,4.812475,4.812475,10.815943,NaN,0.0,...,229.290984,0.0,0.0,13218.526352,1983.364754,0.0,1967.905738,0.0,NaN,47.500000
std,NaN,NaN,36.250842,64.482135,59.705778,3.337248,3.337248,7.938379,NaN,0.0,...,246.981384,0.0,0.0,14353.049405,2184.018619,0.0,2172.072420,0.0,NaN,20.615528
min,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,NaN,20.000000
25%,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0,...,0.000000,0.0,0.0,0.000000,1.000000,0.0,0.000000,0.0,NaN,42.500000
50%,NaN,NaN,74.000000,122.000000,109.000000,6.060000,6.060000,13.246500,NaN,0.0,...,132.500000,0.0,0.0,8204.895000,1298.500000,0.0,1294.000000,0.0,NaN,50.000000
75%,NaN,NaN,82.000000,134.000000,131.000000,7.540250,7.540250,16.410000,NaN,0.0,...,382.250000,0.0,0.0,22735.385000,3179.250000,0.0,3180.250000,0.0,NaN,55.000000


In [5]:
# Select only numeric columns to avoid dtype errors
numeric_columns = df_session.select_dtypes(include='number')

# Identify columns with zero standard deviation
zero_variance_columns = numeric_columns.columns[numeric_columns.std() == 0]
df_session.drop(columns=zero_variance_columns, inplace=True)

# Identify columns with only one unique value
single_value_columns = df_session.columns[df_session.nunique() == 1]

# Drop those columns from the DataFrame
df_session.drop(columns=single_value_columns, inplace=True)

df_session.describe(include = 'all')

,source_file,avg_cadence,avg_heart_rate,avg_power,avg_speed,enhanced_avg_speed,enhanced_max_speed,max_cadence,max_heart_rate,max_power,max_speed,num_laps,start_time,timestamp,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time,unknown_193
count,244,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,173,244,244.000000,244.000000,244.000000,244.000000,244.000000,4.000000
unique,244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,173,244,NaN,NaN,NaN,NaN,NaN,NaN
top,2026-01-10-13-32-12.fit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-10 12:33:03,2026-01-10 12:51:14,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,53.909836,85.368852,85.077869,4.812475,4.812475,10.815943,69.590164,103.504098,217.401639,10.815943,6.405738,NaN,NaN,134.561475,229.290984,13218.526352,1983.364754,1967.905738,47.500000
std,NaN,36.250842,64.482135,59.705778,3.337248,3.337248,7.938379,46.394844,78.734356,192.491768,7.938379,9.223969,NaN,NaN,206.821806,246.981384,14353.049405,2184.018619,2172.072420,20.615528
min,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
25%,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,NaN,NaN,0.000000,0.000000,0.000000,1.000000,0.000000,42.500000
50%,NaN,74.000000,122.000000,109.000000,6.060000,6.060000,13.246500,95.500000,145.000000,199.000000,13.246500,1.000000,NaN,NaN,61.000000,132.500000,8204.895000,1298.500000,1294.000000,50.000000
75%,NaN,82.000000,134.000000,131.000000,7.540250,7.540250,16.410000,103.000000,164.000000,302.000000,16.410000,10.000000,NaN,NaN,182.000000,382.250000,22735.385000,3179.250000,3180.250000,55.000000


In [7]:
df_session.drop(columns=['enhanced_avg_speed','enhanced_max_speed','num_laps','unknown_193'], inplace=True)
df_session.describe(include = 'all')

,source_file,avg_cadence,avg_heart_rate,avg_power,avg_speed,max_cadence,max_heart_rate,max_power,max_speed,start_time,timestamp,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time
count,244,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,173,244,244.000000,244.000000,244.000000,244.000000,244.000000
unique,244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,173,244,NaN,NaN,NaN,NaN,NaN
top,2026-01-10-13-32-12.fit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-10 12:33:03,2026-01-10 12:51:14,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
mean,NaN,53.909836,85.368852,85.077869,4.812475,69.590164,103.504098,217.401639,10.815943,NaN,NaN,134.561475,229.290984,13218.526352,1983.364754,1967.905738
std,NaN,36.250842,64.482135,59.705778,3.337248,46.394844,78.734356,192.491768,7.938379,NaN,NaN,206.821806,246.981384,14353.049405,2184.018619,2172.072420
min,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,1.000000,0.000000
50%,NaN,74.000000,122.000000,109.000000,6.060000,95.500000,145.000000,199.000000,13.246500,NaN,NaN,61.000000,132.500000,8204.895000,1298.500000,1294.000000
75%,NaN,82.000000,134.000000,131.000000,7.540250,103.000000,164.000000,302.000000,16.410000,NaN,NaN,182.000000,382.250000,22735.385000,3179.250000,3180.250000


In [8]:
df_session.head(15)

,source_file,avg_cadence,avg_heart_rate,avg_power,avg_speed,max_cadence,max_heart_rate,max_power,max_speed,start_time,timestamp,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time
0,2026-01-10-13-32-12.fit,78,0,75,5.773,99,0,189,11.254,2026-01-10 12:33:03,2026-01-10 12:51:14,16,69,5606.35,970.0,971.0
1,2026-01-10-13-58-48.fit,0,0,0,0.000,0,0,0,0.000,NaN,2026-01-10 12:59:20,0,0,0.00,1.0,0.0
2,2026-01-10-13-59-21.fit,53,0,75,4.594,88,0,485,13.917,2026-01-10 13:14:42,2026-01-10 14:45:36,199,403,24694.82,5374.0,5375.0
3,2026-01-10-17-37-48.fit,64,0,92,6.107,92,0,324,16.685,2026-01-10 16:38:43,2026-01-10 17:10:21,85,112,7780.89,1289.0,1274.0
4,2026-01-11-11-16-39.fit,44,0,76,5.676,66,0,202,11.243,2026-01-11 10:17:25,2026-01-11 10:45:31,56,96,7458.55,1415.0,1314.0
5,2026-01-11-12-00-28.fit,81,0,47,4.061,121,0,126,10.133,2026-01-11 11:01:28,2026-01-11 11:34:26,33,83,7506.25,1913.0,1848.0
6,2026-01-12-17-33-19.fit,66,0,59,5.178,86,0,129,9.417,2026-01-12 16:34:12,2026-01-12 16:40:05,8,20,1817.82,350.0,351.0
7,2026-01-12-17-41-44.fit,67,0,93,6.208,94,0,241,10.939,2026-01-12 16:41:53,2026-01-12 17:05:10,33,122,8431.43,1357.0,1358.0
8,2026-01-14-16-32-36.fit,73,0,105,6.198,103,0,211,16.268,2026-01-14 15:32:55,2026-01-14 16:06:16,87,184,11344.02,1829.0,1830.0
9,2026-01-15-14-41-29.fit,84,0,96,4.436,141,0,305,22.358,2026-01-15 13:41:48,2026-01-15 14:03:42,99,119,5776.04,1314.0,1302.0


In [10]:
df_session.rename(columns={"timestamp": "end_time"}, inplace=True)

In [13]:
cols = df_session.columns.tolist()
cols.remove("start_time")
cols.remove("end_time")
cols.insert(1, "start_time")
cols.insert(2, "end_time")

df_session = df_session[cols]

In [14]:
df_session.sample(5)

,source_file,start_time,end_time,avg_cadence,avg_heart_rate,avg_power,avg_speed,max_cadence,max_heart_rate,max_power,max_speed,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time
141,2026-04-30-17-37-19.fit,2026-04-30 15:41:50,2026-04-30 16:40:21,79,131,127,6.626,100,152,249,15.860,260,412,22528.79,3487.0,3400.0
168,2026-05-14-17-00-15.fit,2026-05-14 15:00:16,2026-05-14 15:09:50,88,122,132,8.472,99,144,177,12.467,12,72,4871.95,574.0,575.0
122,2026-04-12-16-55-18.fit,2026-04-12 15:00:17,2026-04-12 15:49:53,81,155,157,8.425,111,192,625,14.959,150,457,24930.44,2958.0,2959.0
238,2026-08-02-15-56-41.fit,2026-08-02 13:59:16,2026-08-02 14:12:12,82,132,131,5.799,91,151,227,9.529,70,97,4471.32,770.0,771.0
11,2026-01-15-15-04-51.fit,2026-01-15 14:14:32,2026-01-15 15:16:48,74,0,117,5.681,113,0,460,11.920,266,427,20554.96,3617.0,3618.0


In [16]:
df_session.end_time.dtype

<StringDtype(na_value=nan)>

In [17]:
df_session["start_time"] = pd.to_datetime(df_session["start_time"], utc=True)
df_session["end_time"] = pd.to_datetime(df_session["end_time"], utc=True)

In [20]:
df_session["start_time_Berlin"] = df_session["start_time"].dt.tz_convert("Europe/Berlin")
df_session["end_time_Berlin"] = df_session["end_time"].dt.tz_convert("Europe/Berlin")

cols = df_session.columns.tolist()
cols.remove("start_time_Berlin")
cols.remove("end_time_Berlin")
cols.insert(3, "start_time_Berlin")
cols.insert(4, "end_time_Berlin")

df_session = df_session[cols]                                                                     

In [21]:
df_session.sample(5)

,source_file,start_time,end_time,start_time_Berlin,end_time_Berlin,avg_cadence,avg_heart_rate,avg_power,avg_speed,max_cadence,max_heart_rate,max_power,max_speed,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time
32,2026-02-03-20-04-03.fit,NaT,2026-02-03 19:04:18+00:00,NaT,2026-02-03 20:04:18+01:00,0,0,0,0.000,0,0,0,0.000,0,0,0.00,1.0,0.0
166,2026-05-14-16-58-43.fit,2026-05-14 14:59:21+00:00,2026-05-14 14:59:22+00:00,2026-05-14 16:59:21+02:00,2026-05-14 16:59:22+02:00,38,88,120,7.032,38,89,120,4.421,0,0,14.06,1.0,2.0
236,2026-07-16-19-09-52.fit,2026-07-16 17:14:27+00:00,2026-07-16 18:01:37+00:00,2026-07-16 19:14:27+02:00,2026-07-16 20:01:37+02:00,78,151,175,8.734,93,186,420,17.540,155,487,24726.20,2830.0,2831.0
133,2026-04-23-19-09-41.fit,2026-04-23 17:14:28+00:00,2026-04-23 18:02:01+00:00,2026-04-23 19:14:28+02:00,2026-04-23 20:02:01+02:00,83,156,155,6.901,99,187,400,16.908,237,438,19697.04,2853.0,2854.0
171,2026-05-16-17-33-53.fit,NaT,2026-05-16 15:34:42+00:00,NaT,2026-05-16 17:34:42+02:00,0,0,0,0.000,0,0,0,0.000,0,0,0.00,1.0,0.0


In [24]:
df_session[df_session.start_time.isna()].describe()

,avg_cadence,avg_heart_rate,avg_power,avg_speed,max_cadence,max_heart_rate,max_power,max_speed,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time
count,71.0,71.0,71.0,71.0,71.0,71.0,71.0,71.0,71.0,71.000000,71.0,71.0,71.0
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.028169,0.0,1.0,0.0
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.237356,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.0,0.0
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.0,0.0
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.0,0.0
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.000000,0.0,1.0,0.0


In [26]:
df_session.dropna(subset=["start_time"], inplace=True)
df_session.describe(include='all')

,source_file,start_time,end_time,start_time_Berlin,end_time_Berlin,avg_cadence,avg_heart_rate,avg_power,avg_speed,max_cadence,max_heart_rate,max_power,max_speed,total_ascent,total_calories,total_distance,total_elapsed_time,total_timer_time
count,173,173,173,173,173,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000,173.000000
unique,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2026-01-10-13-32-12.fit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2026-04-11 16:57:39.606936+00:00,2026-04-11 17:44:56.843930+00:00,2026-04-11 18:57:39.606936+02:00,2026-04-11 19:44:56.843930+02:00,76.034682,120.404624,119.994220,6.787538,98.150289,145.982659,306.624277,15.254855,189.786127,323.381503,18643.470694,2796.936416,2775.543353
min,NaN,2026-01-10 12:33:03+00:00,2026-01-10 12:51:14+00:00,2026-01-10 13:33:03+01:00,2026-01-10 13:51:14+01:00,21.000000,0.000000,24.000000,2.232000,34.000000,0.000000,24.000000,1.534000,0.000000,0.000000,5.850000,0.000000,1.000000
25%,NaN,2026-02-21 08:14:39+00:00,2026-02-21 08:55:24+00:00,2026-02-21 09:14:39+01:00,2026-02-21 09:55:24+01:00,73.000000,121.000000,108.000000,5.799000,94.000000,142.000000,196.000000,12.573000,48.000000,122.000000,7349.660000,1080.000000,1065.000000
50%,NaN,2026-04-08 12:14:16+00:00,2026-04-08 12:46:53+00:00,2026-04-08 14:14:16+02:00,2026-04-08 14:46:53+02:00,79.000000,129.000000,121.000000,7.017000,99.000000,155.000000,243.000000,15.570000,133.000000,286.000000,16361.950000,2408.000000,2409.000000
75%,NaN,2026-05-25 07:10:08+00:00,2026-05-25 07:10:10+00:00,2026-05-25 09:10:08+02:00,2026-05-25 09:10:10+02:00,83.000000,141.000000,138.000000,7.980000,106.000000,177.000000,411.000000,17.549000,237.000000,448.000000,25661.670000,3686.000000,3652.000000
max,NaN,2026-08-04 15:14:20+00:00,2026-08-04 15:55:43+00:00,2026-08-04 17:14:20+02:00,2026-08-04 17:55:43+02:00,93.000000,182.000000,200.000000,9.431000,141.000000,203.000000,758.000000,29.242000,1539.000000,1027.000000,61755.870000,8792.000000,8793.000000


Der DataFrame df_temperature enthält Messdaten der Station München-Stadt für den Zeitraum vom 10. Februar 2025 bis zum 13. August 2025.

Jeder Datenpunkt umfasst die innerhalb der letzten zehn Minuten gemessene Höchst- und Tiefsttemperatur. Die Messungen wurden in einer Höhe von 2 Metern sowie 5 Zentimetern über dem Boden durchgeführt.

Für die weitere Analyse wird ausschließlich die Lufttemperatur in 2 Metern Höhe verwendet, da diese die Temperatur in meiner Wohnung vermutlich stärker beeinflusst. Die entsprechenden Werte sind in den Spalten TX_10 (Höchsttemperatur) und TN_10 (Tiefsttemperatur) enthalten.

In [ ]:
df_temperature["TM_10"] = (df_temperature["TX_10"] + df_temperature["TN_10"])/2
print(df_temperature)

In [ ]:
df_temperature.drop(columns=['TX5_10', 'TN5_10', 'eor'], inplace=True)
print(df_temperature)

Die Fitnessdateien enthalten für jede Sekunde eines Workouts einen Datenpunkt. Dabei werden zahlreiche Messwerte in unterschiedlichen Spalten gespeichert. Einige dieser Spalten werden von Zwift jedoch nicht befüllt und enthalten daher keine relevanten Daten.

Im Folgenden wird zunächst ein kurzer Überblick über die vorhandenen Spalten gegeben.


In [ ]:
print(df_fitness.describe())

In [ ]:
print(df_fitness.count() == 0)

In [ ]:
df_fitness.dropna(axis=1, how='all', inplace = True)
print(df_fitness.head())

In [ ]:
print(df_fitness[df_fitness['altitude'] != df_fitness['enhanced_altitude']])
print(df_fitness['speed_1s'].unique())

In [ ]:
df_fitness.drop(columns = ['enhanced_altitude', 'speed_1s'], inplace = True)
print(df_fitness.head())

In [ ]:
df_fitness.info()

In [ ]:
df_temperature.info()

In [ ]:
df_temperature.columns

In [ ]:
df_temperature.columns = df_temperature.columns.str.strip()
df_temperature.columns

In [ ]:
# 1. Datumsformate in echte datetime-Objekte umwandeln
df_temperature['MESS_DATUM'] = pd.to_datetime(df_temperature['MESS_DATUM'], format='%Y%m%d%H%M', utc=True)
df_fitness['timestamp'] = pd.to_datetime(df_fitness['timestamp'], format='%Y-%m-%d %H:%M:%S', utc=True)

# 2. Beide DataFrames nach der Zeit sortieren (Voraussetzung für merge_asof)
df_temperature = df_temperature.sort_values('MESS_DATUM')
df_fitness = df_fitness.sort_values('timestamp')

# 3. merge_asof: findet für jeden fitness-Zeitpunkt den passenden
#    MESS_DATUM-Wert, der <= timestamp ist (backward), 
#    aber maximal 10 Minuten zurückliegt (tolerance)
df_merged = pd.merge_asof(
    df_fitness,
    df_temperature[['MESS_DATUM', 'QN', 'TM_10']],
    left_on='timestamp',
    right_on='MESS_DATUM',
    direction='forward',
    tolerance=pd.Timedelta('10min')
)

In [ ]:
df_merged.sample(10)

In [ ]:
df_merged.describe()

In [ ]:
df_merged[df_merged.TM_10 == -999]

In [ ]:
df_merged["heart_rate"] = df_merged["heart_rate"].replace(0, np.nan)
df_merged["TM_10"] = df_merged["TM_10"].replace(-999, np.nan)
df_merged["target_power"] = df_merged["target_power"].replace(0, np.nan)

In [ ]:
df_merged.describe()

Bei der Tageszeit muss beachtet werden, dass timestamp in UTC angegeben ist

In [ ]:
df_merged['timestamp_berlin'] = df_merged['timestamp'].dt.tz_convert('Europe/Berlin')

In [ ]:
df_merged["day_of_week"] = df_merged.timestamp_berlin.dt.day_name()
df_merged.sample(5)

In [ ]:
import plotly.express as px

df_merged['hour_utc'] = df_merged['timestamp'].dt.hour

fig = px.histogram(df_merged, x='hour_utc', nbins=24, 
                    title='Verteilung nach Stunde')
fig.update_xaxes(dtick=1, title='Stunde des Tages')
fig.update_yaxes(title='Anzahl')
fig.show()

In [ ]:
df_merged['hour_berlin'] = df_merged['timestamp_berlin'].dt.hour

fig = px.histogram(df_merged, x='hour_berlin', nbins=24, 
                    title='Verteilung nach Stunde')
fig.update_xaxes(dtick=1, title='Stunde des Tages')
fig.update_yaxes(title='Anzahl')
fig.show()

In [ ]:
# Beispiel-Daten (ersetze 'timestamp' mit deiner Datums-/Zeit-Spalte)
df['time_of_day'] = df['timestamp'].dt.hour.apply(
    lambda hour:
        'Morning' if 6 <= hour < 12 else
        'Afternoon' if 12 <= hour < 18 else
        'Evening' if 18 <= hour < 22 else
        'Night'
)